<style>
.nvidia-banner {background:#171717;color:white;border-left:10px solid #76b900;padding:18px 22px}
.checkpoint {border:1px solid #aaa;padding:10px 14px;background:#fafafa}
</style>
<div class="nvidia-banner"><h1>Module 3 — Agent-Guided nvMolKit Panel Design</h1>
ACS Fall 2026 · Bounded scientific workflow</div>

## Mission

Select exactly **24 compounds** from the fixed **96-row ReFRAME teaching snapshot**. Match or improve the first 24 stable source rows on structural separation and descriptor coverage, with at least one strict improvement.

## Roles and modes

Python owns both allow-listed executable strategies, exact source rendering, child execution, and artifact validation. Hosted Nemotron returns only a strict plan and strict audit. Reference mode uses the same controller with zero client calls and no key.

The accelerated path uses nvMolKit on a compatible NVIDIA GPU. The small RDKit CPU path is a labeled reference and is never GPU-performance evidence. Set `NVMOLKIT_WORKSHOP_MODE=reference` before kernel start for the direct key-free path.

In [ ]:
from pathlib import Path
import importlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem, RDLogger, rdBase
from rdkit.Chem import Draw

from workshop_common import add_descriptors, load_reframe
import workshop_llm_agent as _workshop_llm_agent

EXPECTED_WORKSHOP_AGENT_VERSION = "2026.08.18.4"
_workshop_llm_agent = importlib.reload(_workshop_llm_agent)
if _workshop_llm_agent.WORKSHOP_AGENT_VERSION != EXPECTED_WORKSHOP_AGENT_VERSION:
    raise RuntimeError("workshop_llm_agent.py does not match this notebook.")

import module3_interactive_workflow as _module3_interactive_workflow

EXPECTED_MODULE3_WORKFLOW_VERSION = "2026.08.18.1"
_module3_interactive_workflow = importlib.reload(_module3_interactive_workflow)
if _module3_interactive_workflow.MODULE3_WORKFLOW_VERSION != EXPECTED_MODULE3_WORKFLOW_VERSION:
    raise RuntimeError("module3_interactive_workflow.py does not match this notebook.")

from workshop_llm_agent import (
    PanelDesignAgent,
    get_workshop_api_key,
    validate_panel_artifacts,
    workshop_mode,
)
from module3_interactive_workflow import launch_interactive_panel_design

WORKSHOP_MODE = workshop_mode()
PANEL_AGENT_MODE = "reference" if WORKSHOP_MODE == "reference" else "hosted"
SEED = 2026
PANEL_SIZE = 24
np.random.seed(SEED)
RDLogger.DisableLog("rdApp.error")
print(
    "Workshop agent:", EXPECTED_WORKSHOP_AGENT_VERSION,
    "| workflow:", EXPECTED_MODULE3_WORKFLOW_VERSION,
    "| mode:", PANEL_AGENT_MODE,
    "| RDKit:", rdBase.rdkitVersion,
)

## Step 1 — Load the fixed input

The default is the bundled snapshot. Ambient `REFRAME_CSV` cannot change it, and the reference path makes no network request.

In [ ]:
candidate_pool = add_descriptors(
    load_reframe(96, source="snapshot")
).reset_index(drop=True)
assert len(candidate_pool) == 96
assert candidate_pool["canonical_ikey"].nunique() == 96
assert candidate_pool["canonical_ikey"].notna().all()
print("Source:", candidate_pool.attrs["source"])
print("Candidate rows:", len(candidate_pool), "| panel rows:", PANEL_SIZE)

## Step 2 — Materialize the bounded workspace

The notebook derives installed data paths from `workshop_llm_agent.__file__`. Outputs use one relative workspace. Source order is preserved because the first 24 rows are the declared baseline.

In [ ]:
MODULE_DIR = Path(_workshop_llm_agent.__file__).resolve().parent
SNAPSHOT_PATH = MODULE_DIR / "data" / "reframe_teaching_snapshot.csv"
assert SNAPSHOT_PATH.is_file()

AGENT_WORKDIR = Path("module3_agent_workspace").resolve()
AGENT_WORKDIR.mkdir(parents=True, exist_ok=True)
input_columns = [
    "smile", "canonical_ikey", "name", "source", "source_id", "status",
    "reframedb_url", "MolWt", "cLogP", "TPSA", "HBD", "HBA", "RotB",
]
candidate_pool[input_columns].to_csv(
    AGENT_WORKDIR / "reframe_candidates.csv", index=False
)

MISSION = """
Design 24 unique compounds from the fixed 96-row ReFRAME teaching snapshot.
The baseline is the first 24 stable source rows. Minimum distance is the
minimum upper-triangle value of 1 minus Tanimoto similarity. Descriptor
coverage is the mean normalized MolWt, cLogP, and TPSA range; a zero candidate
range contributes 1.0. Match or improve both baseline metrics and strictly
improve at least one. Report no biological-performance claim.
""".strip()
(AGENT_WORKDIR / "MISSION.md").write_text(MISSION + "\n", encoding="utf-8")
print("Workspace:", AGENT_WORKDIR)

## Step 3 — Define the artifact gate and shared renderer

These functions are defined before either launch path. They validate and render the same current artifacts for a direct reference run or a later hosted completion callback.

In [ ]:
panel = None
report = None
agent_trace = None
agent_run = None
module3_workflow = None


def load_validated_panel_artifacts(run):
    """Load only one successful result after independent validation."""
    if run is None or not run.success:
        raise ValueError("A successful validated panel run is required.")
    receipt = validate_panel_artifacts(
        AGENT_WORKDIR, expected_panel_size=PANEL_SIZE
    )
    trace_path = AGENT_WORKDIR / "agent_trace.json"
    if trace_path.is_symlink() or not trace_path.is_file():
        raise ValueError("agent_trace.json must be a regular workspace file.")
    loaded_panel = pd.read_csv(AGENT_WORKDIR / "panel.csv")
    loaded_report = json.loads(
        (AGENT_WORKDIR / "report.json").read_text(encoding="utf-8")
    )
    loaded_trace = json.loads(trace_path.read_text(encoding="utf-8"))
    if loaded_trace.get("success") is not True:
        raise ValueError("agent_trace.json does not record a successful run.")
    loaded_panel = loaded_panel.merge(
        candidate_pool[["canonical_ikey", "_mol"]],
        on="canonical_ikey",
        how="left",
        validate="one_to_one",
    )
    if loaded_panel["_mol"].isna().any():
        raise ValueError("Panel rows did not map to candidate molecules.")
    return loaded_panel, loaded_report, loaded_trace, receipt


def render_validated_panel_run(run):
    """Render the same validated artifacts in reference or hosted mode."""
    global panel, report, agent_trace, agent_run
    panel, report, agent_trace, receipt = load_validated_panel_artifacts(run)
    agent_run = run
    acceptance = report["acceptance"]
    summary = {
        "mode": agent_trace["mode"],
        "candidate_count": receipt["candidate_count"],
        "panel_count": receipt["panel_count"],
        "strict_subset": receipt["strict_subset"],
        "baseline_minimum_distance": acceptance["baseline_minimum_distance"],
        "selected_minimum_distance": acceptance["selected_minimum_distance"],
        "baseline_descriptor_coverage": acceptance["baseline_descriptor_coverage"],
        "selected_descriptor_coverage": acceptance["selected_descriptor_coverage"],
        "acceptance_passed": receipt["acceptance_passed"],
        "backend": report["backend"],
    }
    print("Validated panel result")
    print(json.dumps(summary, indent=2, sort_keys=True))
    print("MODULE3_REPORT_JSON=" + json.dumps(summary, sort_keys=True))

    figure, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    for axis, column in zip(axes, ["MolWt", "cLogP", "TPSA"]):
        axis.hist(
            candidate_pool[column], bins=18, density=True, alpha=0.35,
            color="#333333", label="96 candidates",
        )
        axis.hist(
            panel[column], bins=12, density=True, alpha=0.70,
            color="#76b900", label="24 selected",
        )
        axis.set(title=column, xlabel=column, ylabel="Density")
    axes[0].legend()
    figure.suptitle("Fixed-candidate descriptor coverage", y=1.03)
    plt.tight_layout()
    plt.show()
    display(
        panel[
            ["selection_order", "name", "MolWt", "cLogP", "TPSA", "selection_reason"]
        ].head(12).round(2)
    )
    return summary

## Step 4 — Plan, approve, and run

Reference mode runs the fixed plan directly. Hosted mode requests the strict plan, displays the approval widget, and returns in a safe waiting state. The hosted model never receives or writes executable source.

In [ ]:
if WORKSHOP_MODE == "reference":
    panel_agent = PanelDesignAgent(
        workdir=AGENT_WORKDIR,
        mission=MISSION,
        mode="reference",
    )
    reference_plan = panel_agent.request_plan()
    print("Reference strategy:", reference_plan.recommended_strategy)
    agent_run = panel_agent.run(
        approved_strategy=reference_plan.recommended_strategy,
        expected_panel_size=PANEL_SIZE,
        max_revisions=0,
        timeout_seconds=180,
    )
    if not agent_run.success:
        raise RuntimeError(agent_run.attempts[-1].message)
    render_validated_panel_run(agent_run)
else:
    agent_api_key = get_workshop_api_key()
    panel_agent = PanelDesignAgent(
        workdir=AGENT_WORKDIR,
        mission=MISSION,
        mode="hosted",
        api_key=agent_api_key,
    )
    module3_workflow = launch_interactive_panel_design(
        panel_agent,
        expected_panel_size=PANEL_SIZE,
        max_revisions=0,
        timeout_seconds=480,
        on_complete=render_validated_panel_run,
    )
    print("Waiting for sponsor approval in the widget; later cells remain safe.")

## Step 5 — Inspect current state

In hosted mode, Run All reaches this section while approval is pending. It does not load stale files and it does not fail.

In [ ]:
if agent_run is None:
    print("Waiting for sponsor approval. No scientific result is loaded yet.")
else:
    display(
        pd.DataFrame(
            [
                {
                    "attempt": attempt.number,
                    "source_file": attempt.source_file,
                    "seconds": attempt.elapsed_seconds,
                    "passed": attempt.passed,
                    "message": attempt.message,
                }
                for attempt in agent_run.attempts
            ]
        )
    )
    print("Trace:", agent_run.trace_path)

## Step 6 — Inspect selected chemistry

This bounded gallery is a qualitative check on the validated receipt. It does not establish binding, activity, ADMET, efficacy, safety, conformation, or clinical value.

In [ ]:
if panel is None:
    print("Waiting for sponsor approval; the chemistry gallery is not available yet.")
else:
    gallery_rows = panel.sort_values(
        ["selection_order", "canonical_ikey"]
    ).head(12)
    gallery = Draw.MolsToGridImage(
        [Chem.Mol(molecule) for molecule in gallery_rows["_mol"]],
        legends=[
            (
                f"{int(row.selection_order)}. {str(row.name)[:24]}\n"
                f"MW {row.MolWt:.0f} | cLogP {row.cLogP:.1f} | TPSA {row.TPSA:.0f}"
            )
            for row in gallery_rows.itertuples()
        ],
        molsPerRow=4,
        subImgSize=(260, 220),
        useSVG=True,
    )
    display(gallery)

## Final synthesis

A passing receipt proves only that the fixed 96 keys were unique, the 24 selected keys were a unique strict subset, both baseline metrics were matched or improved, and at least one improved strictly.

Minimum distance is the minimum upper-triangle value of `1 - Tanimoto similarity`. Descriptor coverage is the mean selected-to-candidate range ratio for MolWt, cLogP, and TPSA; a zero candidate range contributes 1.0.

<div class="checkpoint"><b>Exit ticket:</b> State which allow-listed strategy you would defend, its representation assumptions, and the next experimental constraint needed before a repurposing claim.</div>

Fingerprints, clusters, descriptors, and 2D drawings do not establish biological performance. The RDKit path is a CPU reference and never nvMolKit GPU-performance evidence.